# 01 — Easy Tensor Node Classification

**Goal:** Train a node classifier on a graph where each node carries a
structured `[C, H, W]` tensor using the TGraphX Easy Mode API.

**Why this matters:** Most graph frameworks treat node features as flat
vectors.  TGraphX keeps tensor structure intact through every message-passing
step.  This notebook shows you can get a working model in a few lines.

**TGraphX subsystem:** `tgraphx.easy`

**Data:** Synthetic — no download required.

**Runtime:** < 30 seconds on CPU.

## 1. Setup

In [ ]:
# Optional: uncomment to install in Colab
# !pip install -q tgraphx
import tgraphx as tgx
print("TGraphX version:", tgx.__version__)

## 2. Scenario

We have a synthetic citation graph.  Each node (paper) is represented by a
`[C, H, W] = [4, 6, 6]` tensor — imagine a small feature map summarising
the paper's topic distribution across semantic categories.

There are 3 topic classes.  Our goal: classify each paper into one class.

## 3. Create Synthetic Data

In [ ]:
# Create a synthetic tensor node classification graph.
# num_nodes=256, node_shape=(4,6,6), 3 classes, no GPU required.
data = tgx.easy.synthetic_tensor_node_classification(
    num_nodes=256,
    node_shape=(4, 6, 6),    # [channels, height, width] per node
    num_classes=3,
    num_edges=1024,
    seed=42,
)
print(f"Graph: {data.num_nodes} nodes, {data.num_edges} edges")
print(f"Node feature shape per node: {data.node_features.shape[1:]}")  # (4,6,6)
print(f"Labels shape: {data.node_labels.shape}")

## 4. Train with Easy Mode

In [ ]:
# Zero-boilerplate training: one call handles model, sampler, optimizer, loss.
result = tgx.easy.train_node_classifier(
    data,
    model="tensor_gcn",   # ConvMessagePassing-based, preserves [C,H,W]
    sampler="neighbor",   # mini-batch NeighborLoader
    fanouts=[10, 5],      # 2-hop sampling
    batch_size=32,
    epochs=5,
    seed=42,
    verbose=True,
)

## 5. Inspect Result

In [ ]:
print("\nFinal metrics:")
for k, v in result.metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nConfig (all resolved defaults visible):")
for k, v in result.config.items():
    print(f"  {k}: {v}")

## 6. Why This Is Tensor-Native

In [ ]:
# Node features are kept as [N, C, H, W] — never flattened.
import torch
print("node_features shape:", result.graph.node_features.shape)
# ↑ (256, 4, 6, 6) — 4 channels, 6×6 spatial layout preserved.

# The model internals: ConvMessagePassing uses 1×1 conv, not Linear(flatten).
print("Model type:", type(result.model).__name__)

# You can always escape to raw PyTorch objects:
print("Optimizer:", type(result.optimizer).__name__)
print("Loader:", type(result.loader).__name__)

## 7. Write Dashboard Artifacts (optional)

If you have a logdir and want to view metrics in `tgraphx-dashboard`:

```python
result.write_dashboard_artifacts("runs/easy_nb01")
# Then: tgraphx-dashboard --logdir runs/easy_nb01
```

## 8. Next Steps

- **Script version:** `examples/easy_tensor_node_classification_no_torch.py`
- **Tutorial:** `tutorials/tensor_node_classification_neighbor_loader.py`
- **Low-level API:** see `tgraphx/easy/_workflows.py` for the full training loop.
- **Limitations:** this is a synthetic demo; real-world accuracy depends on
  your dataset and training length.